# SmileAlign v2, YOLOv8n-seg campuran intraoral + AlphaDent (1 kelas `tooth`)

Dataset sanjungan dua domain:
- **Intraoral Tooth Detection** (Roboflow, CC BY 4.0): `smilealign/intraoral_yolov8.zip`
- **AlphaDent** (Apache-2.0, diunduh dari HuggingFace): 1.455 foto gigi DSLR

Semua kelas disatukan ke `tooth` karena backend hanya memakai geometri instance.

## Cara pakai

1. Pastikan `intraoral_yolov8.zip` ada di `MyDrive/smilealign/`.
2. Opsional: taruh `best.pt` v1 di `smilealign/best.pt` sebagai warm start.
3. Runtime > Change runtime type > **GPU T4**.
4. Run All. Latihan sekitar 1 jam (imgsz 960).
5. Unduh `best.pt` ke `backend/models/seg/best.pt`.

## Kalau sesi terputus

Jalankan Sel 5. Sel itu membaca epoch terakhir dari `results.csv`, memulihkan
`last.pt` dari cadangan, menyelaraskan argumen dengan versi Ultralytics yang
terpasang, lalu memanggil `train(resume=True)`.

`/content` hilang saat runtime mati, jadi Sel 1 dan Sel 2 perlu diulang sebelum
Sel 5.

## Sel 0, mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Sel 1, ekstrak intraoral dan unduh AlphaDent

In [ ]:
import os
import zipfile

INTRAORAL_ZIP = "/content/drive/MyDrive/smilealign/intraoral_yolov8.zip"
INTRAORAL_DIR = "/content/datasets/intraoral"
ALPHA_DIR = "/content/datasets/alphadent"

os.makedirs(INTRAORAL_DIR, exist_ok=True)
with zipfile.ZipFile(INTRAORAL_ZIP) as z:
    for info in z.infolist():
        name = info.filename.replace("\\", "/")
        target = os.path.join(INTRAORAL_DIR, name)
        if name.endswith("/"):
            os.makedirs(target, exist_ok=True)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(info) as src, open(target, "wb") as dst:
            dst.write(src.read())
print("intraoral OK:", os.listdir(os.path.join(INTRAORAL_DIR, "intraoral")))

from huggingface_hub import hf_hub_download

zip_path = hf_hub_download(
    repo_id="ZFTurbo/AlphaDent",
    filename="AlphaDent.zip",
    repo_type="dataset",
    local_dir="/content/ad_zip",
)
print("AlphaDent zip:", zip_path)
os.makedirs(ALPHA_DIR, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(ALPHA_DIR)


def has_struct(root):
    a = os.path.isdir(os.path.join(root, "images", "train")) and os.path.isdir(
        os.path.join(root, "labels", "train")
    )
    b = os.path.isdir(os.path.join(root, "train", "images")) and os.path.isdir(
        os.path.join(root, "train", "labels")
    )
    return bool(a or b)


AD_ROOT = None
for root, dirs, files in os.walk(ALPHA_DIR):
    if has_struct(root):
        AD_ROOT = root
        break
assert AD_ROOT, "struktur AlphaDent tidak ketemu, cek isi zip"
print("AD_ROOT:", AD_ROOT)

## Sel 2, gabungkan jadi satu kelas

In [ ]:
import shutil
from pathlib import Path

MIX_DIR = Path("/content/datasets/mix_v2")

JOBS = [
    ("io_", Path(INTRAORAL_DIR) / "intraoral"),
    ("ad_", Path(AD_ROOT)),
]
SPLITS = [("train", "train"), ("valid", "valid"), ("test", "valid")]


def pick_dirs(root, split):
    imgs = [p for p in (root / split / "images", root / "images" / split) if p.is_dir()]
    lbls = [p for p in (root / split / "labels", root / "labels" / split) if p.is_dir()]
    return (imgs[0] if imgs else None), (lbls[0] if lbls else None)


def normalize_row(line):
    # Samakan baris label ke format polygon, kelas selalu 0.
    # AlphaDent mencampur baris polygon dan bbox. Ultralytics menolak file yang
    # mencampur keduanya, jadi bbox diubah menjadi polygon empat titik.
    parts = line.split()
    if len(parts) < 2:
        return None
    coords = parts[1:]
    if len(coords) == 4:
        cx, cy, bw, bh = (float(v) for v in coords)
        x1, y1, x2, y2 = cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2
        corners = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
        return "0 " + " ".join(f"{v:.6f}" for pt in corners for v in pt)
    if len(coords) >= 6 and len(coords) % 2 == 0:
        return "0 " + " ".join(coords)
    return None


counts = {"train": 0, "valid": 0}
dropped = 0
for prefix, root in JOBS:
    for split_src, split_dst in SPLITS:
        imgs, lbls = pick_dirs(root, split_src)
        if imgs is None or lbls is None:
            continue
        (MIX_DIR / split_dst / "images").mkdir(parents=True, exist_ok=True)
        (MIX_DIR / split_dst / "labels").mkdir(parents=True, exist_ok=True)
        for img in sorted(imgs.iterdir()):
            if img.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            lbl = lbls / (img.stem + ".txt")
            if not lbl.exists():
                continue
            rows = []
            for line in lbl.read_text().splitlines():
                t = line.strip()
                if not t:
                    continue
                row = normalize_row(t)
                if row is None:
                    dropped += 1
                    continue
                rows.append(row)
            if not rows:
                continue
            shutil.copy(img, MIX_DIR / split_dst / "images" / f"{prefix}{img.name}")
            (MIX_DIR / split_dst / "labels" / f"{prefix}{img.stem}.txt").write_text(
                "\n".join(rows) + "\n"
            )
            counts[split_dst] += 1

(MIX_DIR / "data.yaml").write_text(
    "train: train/images\nval: valid/images\nnc: 1\nnames: ['tooth']\n"
)
print("merged:", counts, "->", MIX_DIR)
if dropped:
    print("baris label dibuang:", dropped)

## Sel 3, pasang Ultralytics

In [ ]:
# Tidak dipin ke versi tertentu. Resume membaca argumen dari checkpoint, jadi
# versi yang terlalu tua akan menolak argumen baru seperti dis, dgrad, quantize.
!pip -q install -U ultralytics huggingface_hub

import ultralytics

print("ultralytics:", ultralytics.__version__)

## Sel 4, latihan dari nol

In [ ]:
import os

from ultralytics import YOLO

V1 = "/content/drive/MyDrive/smilealign/best.pt"
init = V1 if os.path.exists(V1) else "yolov8n-seg.pt"
print("Init dari:", init)

model = YOLO(init)
results = model.train(
    data=str(MIX_DIR / "data.yaml"),
    epochs=120,
    imgsz=960,
    bat=16,
    device=0,
    project="/content/drive/MyDrive/smilealign/runs-smilealign-seg-v2",
    name="train",
    patience=20,
    cache=False,
    workers=2,
)
metrics = model.val()
print(metrics.results_dict)

## Sel 5, lanjutkan latihan yang terputus

In [ ]:
import shutil
from pathlib import Path

import pandas as pd
import torch

ROOT = Path("/content/drive/MyDrive/smilealign/runs-smilealign-seg-v2")
runs = [p for p in ROOT.glob("train*") if (p / "results.csv").exists()]
assert runs, f"tidak ada results.csv di {ROOT}"

RUN_DIR = max(runs, key=lambda p: (p / "results.csv").stat().st_mtime)
df = pd.read_csv(RUN_DIR / "results.csv")
df.columns = df.columns.str.strip()
print("run   :", RUN_DIR)
print("epoch :", len(df))

weights = RUN_DIR / "weights"
current = weights / "last.pt"
backup = weights / "last_orig.pt"

# Cadangan dibuat sekali, sebelum ada perubahan apa pun pada checkpoint.
if not backup.exists():
    shutil.copy2(current, backup)
    print("cadangan dibuat:", backup.name)
else:
    shutil.copy2(backup, current)
    print("dipulihkan dari:", backup.name)

from ultralytics.cfg import DEFAULT_CFG_DICT

ckpt = torch.load(current, map_location="cpu", weights_only=False)
args = ckpt["train_args"]
known = set(DEFAULT_CFG_DICT)

unknown = sorted(k for k in args if k not in known)
for k in unknown:
    args.pop(k)

coerced = []
for k, v in list(args.items()):
    if isinstance(DEFAULT_CFG_DICT.get(k), bool) and not isinstance(v, bool):
        args[k] = bool(v)
        coerced.append(f"{k}: {v!r} -> {args[k]}")

ckpt["train_args"] = args
torch.save(ckpt, current)
print("dibuang  :", unknown or "(tidak ada)")
print("disamakan:", coerced or "(tidak ada)")

from ultralytics import YOLO

model = YOLO(str(current))
model.train(resume=True)

## Selesai

Hasil ada di Drive, folder `smilealign/runs-smilealign-seg-v2/`. Bila folder
`train` sudah ada, Ultralytics memakai `train-2`, `train-3`, dan seterusnya.
Cek folder terbaru, unduh `weights/best.pt`, taruh di `backend/models/seg/best.pt`.

Referensi sitasi: AlphaDent (Sosnin dkk., 2025, Apache-2.0) + Intraoral Tooth
Detection (CC BY 4.0).